In [15]:
!pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]>=1.112" "google-api-core>=2.19.0"

In [1]:
from google.colab import auth
auth.authenticate_user(project_id="qwiklabs-gcp-00-117e2d1e6738")

In [32]:
import vertexai
from vertexai.generative_models import GenerativeModel
import os

project_id = 'qwiklabs-gcp-00-117e2d1e6738'
location = 'us-central1' # Infrastructure location
staging_bucket = 'gs://exchange_rate_bucket_app_test'

# Set the model location to global as requested
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'

# Initialize Vertex AI
vertexai.init(
    project=project_id,
    location=location,
    staging_bucket=staging_bucket
)

In [33]:
def get_exchange_rate(
    currency_from: str = "USD",
    currency_to: str = "SEK",
    currency_date: str = "latest",
) -> dict:
    """
    Get the exchange rate between two currencies on a specified date.
    """
    import requests
    import json

    url = f"https://api.frankfurter.app/{currency_date}"
    params = {"from": currency_from, "to": currency_to}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        # Explicitly ensure we return a JSON-serializable dictionary
        return dict(data) if data else {"error": "No data"}
    except Exception as e:
        return {"error": str(e)}

In [34]:
from google.adk.agents import Agent
from vertexai import agent_engines

agent = Agent(
    model = "gemini-3.6-flash",
    name = "Get_Exchange_Rate",
    tools = [get_exchange_rate],
)

app = agent_engines.AdkApp(
    agent=agent
)

In [35]:
async for event in app.async_stream_query(
    user_id="test_user_id",
    message="What is the exchange rate from US dollars to SEK today?"
):
  print(event)

{'model_version': 'gemini-3.6-flash', 'content': {'parts': [{'function_call': {'id': 'call_202737', 'args': {'currency_date': 'latest', 'currency_to': 'SEK', 'currency_from': 'USD'}, 'name': 'get_exchange_rate'}, 'thought_signature': 'AY89a1_tyS-jVriNe1lLGKegcBvDPiOjp9DejVd5SxNNsiiHDKV503GHVXFINOobXhFaPWFs8hv7FQnimb8GKHe_n8CiVtwbpv6SYf7jU3NxD18GszOs6rSaKtKOF2E0S7A6NfkS2R9eFoG0zXSRuf-l2AMdOCA2yVrLWJ0aV07NQstRDflT9I68_ogHDXI5ILSuHqgfehP0CAnpI4ai_OCBfpx0ATkI7ODFsxWX8xAeuVfMnK8j2gN-Roio0tIs6_4X6WatH6XyLOYBkqrtMFWR_-UF5MH-C8_ixrAtGq_4VOKNab2B1EwbWOb9rkaHTwywKXKeqew_ccOcYGaWKhPItIDtG8OU9K67b5nuM7UobHIuJ-yBTG_2hlyzb70m_ukEFs5uylFg4T8vJ98Ci-o5PULNXxA-eha_fRFsoBQLfGknBzzc6MdelsBBpAaL6kp1FIr4btaUxUjI8tEDMuxYZrT7_hg5fIvmfpNNFu7Jgv_Bro7lmsWgbHUBtAmNCJiThaCNRzAOkYQdlHBj1fLA4ghLBUFDWJiJGzKiRAV4jZycbojINLYOCgM9hJhBB-g6qDaQHth0mWR6MTdaFKNk3a_rell_iNdjmXYkpfrZ_zZvLPMIrwy0PDelkpUi_8r-fdszeUZZIA=='}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 37, 'candidates_

In [ ]:
from vertexai import agent_engines
import os

# Ensure environment variables are set
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'

# Define deployment requirements
requirements=[
    "google-adk==1.18.0",
    "google-cloud-aiplatform[agent_engines]>=1.112.0",
    "google-genai>=1.9.0",
    "requests",
    "cloudpickle==3.1.2",
    "pydantic==2.13.4"
]

# Deploying with explicit configuration for the reasoning engine
try:
    remote_agent = agent_engines.create(
        app,
        display_name="exchange-rate-agent-final",
        requirements=requirements,
        env_vars={"GOOGLE_CLOUD_LOCATION": "global"},
    )
    print(f"Agent deployment successfully started: {remote_agent.resource_name}")
except Exception as e:
    print(f"Deployment failed: {e}")

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'pydantic': '2.13.4', 'google-cloud-aiplatform': '1.165.1'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-adk==1.18.0', 'google-cloud-aiplatform[agent_engines]>=1.112.0', 'google-genai>=1.9.0', 'requests', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai.agent_engines:Using bucket exchange_rate_bucket_app_test
INFO:vertexai.agent_engines:Wrote to gs://exchange_rate_bucket_app_test/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://exchange_rate_bucket_app_test/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://exchange_rate_bucket_app_test/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/843671275568/locations/us-central1/reasoningEngines/7077329298

In [26]:
try:
    async for event in remote_agent.async_stream_query(
        user_id="test_user_id",
        message="What is the exchange rate from US dollars to SEK today?",
    ):
        print(event)
except Exception as e:
    print(f"An error occurred during remote execution: {e}")
    print("Please check the Google Cloud Logs for your Reasoning Engine to see the full traceback.")

{'error_code': 'TypeError', 'error_message': "'NoneType' object is not subscriptable", 'invocation_id': 'e-68b61e39-b1ca-49bb-a820-c4ee7a193605', 'author': 'Get_Exchange_Rate', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': 'Get_Exchange_Rate@1'}, 'id': '706fffe3-60c7-4163-b1c7-93498894b965', 'timestamp': 1787343908.052152}
{'code': 498, 'message': "'NoneType' object is not subscriptable"}
